# Describe observations with SOSA

Define the part of [SOSA](https://www.w3.org/TR/vocab-ssn/) your project uses. This example describes two pressure observations from the same sensor and groups them into an [observation collection](https://www.w3.org/TR/vocab-ssn-ext/#sosa:ObservationCollection). Each result carries a numeric value and a [QUDT](https://www.qudt.org/doc/2024/12/DOC_SCHEMA-QUDT.html) unit IRI.

Run the cells in order in the environment described in [Scientific metadata with Ontolutils](../Classes.ipynb). This notebook runs offline.

In [ ]:
from pydantic import Field
from ontolutils import QUDT_UNIT, Thing, namespaces, urirefs
from ontolutils.typing import AnyIri

SOSA = "http://www.w3.org/ns/sosa/"
QUDT = "http://qudt.org/schema/qudt/"


@namespaces(qudt=QUDT)
@urirefs(QuantityValue="qudt:QuantityValue", value="qudt:numericValue", unit="qudt:unit")
class QuantityValue(Thing):
    value: float
    unit: AnyIri


@namespaces(sosa=SOSA)
@urirefs(
    Observation="sosa:Observation",
    sensor="sosa:madeBySensor",
    property_observed="sosa:observedProperty",
    result="sosa:hasResult",
)
class Observation(Thing):
    sensor: AnyIri
    property_observed: AnyIri
    result: QuantityValue


@namespaces(sosa=SOSA)
@urirefs(ObservationCollection="sosa:ObservationCollection", members="sosa:hasMember")
class ObservationCollection(Thing):
    members: list[Observation] = Field(min_length=1)

The project chooses the Python field names and the required fields. The sensor and the observed property are references to metadata managed elsewhere. The result and collection members are embedded models, so their statements appear in the serialized graph.

In [ ]:
sensor = "https://example.org/sensors/pressure"
pressure = "https://example.org/observable-properties/static-pressure"

observations = [
    Observation(
        id=f"https://example.org/observations/{index}",
        sensor=sensor,
        property_observed=pressure,
        result=QuantityValue(value=value, unit=QUDT_UNIT.PA),
    )
    for index, value in enumerate([100_000.0, 100_200.0], start=1)
]
collection = ObservationCollection(
    id="https://example.org/collections/pressure",
    members=observations,
)
print(collection.serialize("ttl"))

In [ ]:
from rdflib import Graph, Namespace, RDF, URIRef

graph = Graph().parse(data=collection.serialize("ttl"), format="ttl")
sosa = Namespace(SOSA)
qudt = Namespace(QUDT)
collection_id = URIRef(collection.id)

assert (collection_id, RDF.type, sosa.ObservationCollection) in graph
assert set(graph.objects(collection_id, sosa.hasMember)) == {
    URIRef(observation.id) for observation in observations
}
for observation in observations:
    result = graph.value(URIRef(observation.id), sosa.hasResult)
    assert graph.value(result, qudt.unit) == QUDT_UNIT.PA
    assert float(graph.value(result, qudt.numericValue)) == observation.result.value